# Machine Learning with the Titanic dataset

Use the most famous dataset on Kaggle — the 1912 Titanic passenger list — to try **predicting the future from past data**.

- Every row was a real person. Keep that in mind.
- Goal today: look at the data yourself, predict which features matter, then train a model and check your intuition.

## Step 1 — Load the data

In [ ]:
import pandas as pd

url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
try:
    df = pd.read_csv(url)
except Exception:
    print("If the download fails, ask your teacher for titanic.csv and upload it to the folder on the left.")
    df = pd.read_csv("titanic.csv")
print(df.shape)
df.head()

## Step 2 — Look at the data yourself first

Before letting a machine predict anything, look with your own eyes. Which columns seem to separate who survived?

- `Survived`: 1 = lived, 0 = died | `Pclass`: ticket class (1 = rich ... 3 = poor) | `Sex` | `Age` | `Fare`

In [ ]:
print("overall survival rate:", round(df["Survived"].mean(), 3))
print("\nby Sex")
print(df.groupby("Sex")["Survived"].mean().round(3))
print("\nby Pclass")
print(df.groupby("Pclass")["Survived"].mean().round(3))

# bin Age into child / teen / adult
df["AgeGroup"] = pd.cut(df["Age"], [0, 12, 18, 200], labels=["child(0-12)", "teen(13-18)", "adult(19+)"])
print("\nby Age group")
print(df.groupby("AgeGroup", observed=True)["Survived"].mean().round(3))

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
df.groupby("Sex")["Survived"].mean().plot.bar(ax=axes[0], title="by Sex")
df.groupby("Pclass")["Survived"].mean().plot.bar(ax=axes[1], title="by Pclass")
df.groupby("AgeGroup", observed=True)["Survived"].mean().plot.bar(ax=axes[2], title="by Age group")
for ax in axes:
    ax.set_ylim(0, 1)
    ax.set_ylabel("survival rate")
plt.tight_layout()
plt.show()

**Observe**: Which gap is biggest? You can read 1912 behaviour ("women and children to the lifeboats first") straight out of the numbers.

## Step 3 — Predict BEFORE you train

Before the machine answers, write down **your own** prediction. This is the most important part of the lesson.

1. Which feature separates survivors best? Rank **Sex / Pclass / Age / Fare** from most to least important.
2. Why? Write one line.

Write your ranking down NOW. In a moment the tree will reveal which feature it actually relied on most — then you can check whether your intuition matched the data. **No peeking at Step 4 first!**

## Step 4 — Train a decision tree

A **decision tree** reaches an answer by asking a chain of yes/no questions (like 20 Questions).

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree

data = df[["Pclass", "Sex", "Age", "Fare", "Survived"]].copy()
data["Sex"] = (data["Sex"] == "female").astype(int)  # female=1, male=0
data["Age"] = data["Age"].fillna(data["Age"].median())

X = data[["Pclass", "Sex", "Age", "Fare"]]
y = data["Survived"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

tree = DecisionTreeClassifier(max_depth=3, random_state=42)
tree.fit(X_train, y_train)
print("train accuracy:", round(tree.score(X_train, y_train), 3))
print("test accuracy: ", round(tree.score(X_test, y_test), 3))

In [ ]:
plt.figure(figsize=(14, 7))
plot_tree(tree, feature_names=["Pclass", "Sex(female=1)", "Age", "Fare"],
          class_names=["died", "survived"], filled=True, fontsize=9)
plt.show()

print("feature importance - did it match YOUR ranking from Step 3?")
for name, imp in sorted(zip(X.columns, tree.feature_importances_), key=lambda t: -t[1]):
    print(f"  {name:6} {round(imp, 3)}")

**Read the tree**: What is the top question? That is the feature the model judged to split survival best. **Did it match the ranking you wrote in Step 3?**

## Step 5 — Experiment: what if the tree is too deep? (overfitting)

Change `max_depth` and watch the train vs test accuracy.

In [ ]:
for depth in [1, 2, 3, 5, 10, None]:
    t = DecisionTreeClassifier(max_depth=depth, random_state=42).fit(X_train, y_train)
    print(f"max_depth={str(depth):>4} | train {t.score(X_train, y_train):.3f} | test {t.score(X_test, y_test):.3f}")

**Questions**
- As the tree gets deeper, what happens to train accuracy? To test accuracy?
- When train rises but test falls, that is **overfitting** (memorising instead of learning).

## Reflection
- Where did you "see" human behaviour in the data?
- If this model were used for real decisions, what could go wrong? (connects to the bias page)
- **Kaggle** (13+) lets you try the same challenge as people worldwide: kaggle.com/c/titanic